# POLSCI 3

# Week 6, Notebook Lecture 1: Quantifying uncertainty using the standard error

Following the example in in-class pdf lecture slides, let's use data from the Dunning and Harrison (2010) study to learn about quantifying uncertainty using standard errors. 



#### Background -- Ethnicity and Politics in Diverse Mali 

Many social scientists believe the politicization of ethnicity in politics can be corrosive to democracy. For example, if ethnicity is politicized, people may engage in *ethnic voting*, in which they vote for in-group candidates and parties aimed at advancing the welfare *only* for their own groups. Yet, there are immensely diverse societies in which ethnicity is not very salient in politics. Mali, a landlocked country in West Africa, is one of them.

In [Dunning and Harrison (2010)](http://www.thaddunning.com/wp-content/uploads/2009/12/CrossCutting_APSR-preprint.pdf)'s paper, the authors study a previously under-studied "cross-cutting cleavage," which means an attribute where "dimensions of identity or interest along which members of the same ethnic group may have diverse allegiances." In particular, they argue that the presence of *cousinage* is one reason why ethnic voting is not as strong in Mali.

So, what is *cousinage*? During the Mali Empire (1230 to about 1600), families formed alliances on the basis of patronyms (i.e., surnames), even if were are of different ethnic or religious backgrounds. Even up to today, in countries such as current-day Mali (as well as Senegal, The Gambia, Guinea, and Burkina Faso), members of different ethnic groups feel close to their so-called _"joking cousins"_ even if they are not from the same group. 

In other words, cousinage **cross-cuts** ethnicity. While people favor their co-ethnics, they also favor their joking cousins, who very often are not in the same ethnic group.

#### About the study

Dunning and Harrison study whether the presence of this cross-cutting cleavage -- "joking cousins" -- helps reduce ethnic voting.

Here is a brief description of the experiment's procedures from their paper:

> To measure the effect of cousinage on voter preferences, we showed videotaped political speeches to experimental subjects, who were recruited through door-to-door canvassing in almost all neighborhoods in the capital city of Bamako. Subjects were told that the candidate in the video was a political independent who was considering launching a campaign for deputy in the National Assembly. [In all trials], we asked subjects to evaluate ... the attractiveness of the candidate (`global_eval`). **The content of speeches viewed by all subjects was identical. The experimental manipulation (`treat_assign`) consisted of what subjects were told about the politician’s last name, which conveys information about both ethnic identity and cousinage ties in Mali.** _(Emphasis and variable names added)_

In the paper, the main treatment conditions are:
  1. Same ethnicity, joking cousin
  2. Same ethnicity, not joking cousin
  3. Different ethnicity, joking cousin
  4. Different ethnicity, not joking cousin

Here, **we are going to work with a subset of the dataset that contains only the subjects assigned to treatment condition 1 (same ethnicity, joking cousin) and treatment condition 4 (different ethnicity, not joking cousin).**  In this simplified dataset, the treatment status will be saved in a single `treat` variable, so that `treat=1` if the subject is assigned to the co-ethnic cousin condition (treatment condition 1 above) and `treat=0` if the subject is assigned to the non-coethnic, non-cousing condition (treatment condition 0 above).

Let's take a look at the data. 

In [ ]:
data_dh <- read.csv('DunningHarrison.csv')
head(data_dh)

Here is a quick rundown of what each column in this dataset means:

- `participantid`: a unique identifier for each subject (each person in the experiment)
- `vote_prefer`: 'On a scale from 1 to 7, how much does this speech make you want to vote for (name of politician)?' 7 = most likely to want to vote for; 1 = least likely to want to vote for
- `treat`: `1` if the subject (meaning person in the experiment) was randomly assigned to be shown a candidate with a surname that mkes the candidate a co-ethnic and a joking cousin; `0` if the subject saw a candidate who was neither a co-ethnic nor a joking cousin
- `actor`: this number indicates which actor each participant watched. Speeches were filmed in July 2008 in NYC, with two Malians actors. (`1`: Mamadou Doumbia; `2`: Bamadou Diallo)
- `female`: equals `1` if the respondent identifies as female, `0` otherwise
- `inscrite`: `1` if subject is registered to vote, `0` otherwise
- `cousin_subject`: `1` if subject **believes** the actor is their joking cousin after viewing the speech, `0` otherwise
- `education`: `1` if the subject received higher education, `0` otherwise
- `vecu_ailleurs`: `1` if subject has lived elsewhere from Bamako, `0` otherwise

We can use a one-way table to look at the distribution of experimental subjects in the treatment condition we are using here.  

In [ ]:
table(data_dh$treat)

As we can see, there are 136 in the treatment (coethnic joking cousin) condition, where <code>treat==1</code>, and 152 subjects in the control (non-coethnic, non-cousin) condition. (You can find the same numbers in Table 1 in the paper).

Now, let's compare voting preferences between subjects assigned to the "coethnic cousin" condition and the "non-coethnic, non-cousin" condition.

To look at the means and their difference directly, let's first use the "old" method of calculating these quantities directly:

In [ ]:
# Calculate treatment effect using "old" method

# First calculate the means in each group

coethnic_cousin_data <- subset(data_dh,treat==1)
coethnic_cousin_ave <- mean(coethnic_cousin_data$vote_prefer)
coethnic_cousin_ave # print the result


In [ ]:
noncoethnic_noncousin_data <- subset(data_dh,treat==0)
noncoethnic_noncousin_ave <- mean(noncoethnic_noncousin_data$vote_prefer)
noncoethnic_noncousin_ave # print the result

# Compare these means to Table 4 in the paper -- you will see they are the same

In [ ]:
# Now calculate the estimated effect of being exposed to the "coethnic cousin" condition, relative to the "non-coethnic non-cousin" condition

effect <- coethnic_cousin_ave - noncoethnic_noncousin_ave
effect # print the result



By now, we know that we can also use the `difference_in_means()` function to see how big of a difference the researchers actually saw between these two experimental conditions (the "coethnic cousin" condition versus the "non-coethnic non-cousin" condition)

In [ ]:
library(estimatr)
difference_in_means(vote_prefer ~ treat, data_dh)

We see the estimated effect is 1.09, the same as we calculated using the old method.

But what are the other values that the `difference_in_means()` function outputs -- and how do they help us quantify uncertainty?  

That's where we'll turn now, focusing in this lecture on the standard error. 

## How big of a difference might we see by chance?

This is a fairly small experiment, though. Might we see a difference this big by chance?

Let's suppose for the moment assume that the assigned ethnicity and cousinage relationship of the candidate actually had no effect, so all the potential outcomes are the same: it doesn't matter whether subjects were assigned to the control (non-coethnic non-cousin) or treatment (co-ethnic cousin) group. 

That means we've already seen all the potential outcomes! (After all, if we see the potential outcome under treatment, we've seen the potential outcome under control, and vice versa: this is because if there is no effect for any subject, the potential outcomes in each condition are the same for a given subject!).  This also means that the *true average treatment effect* is zero.

In this world, we can examine how the *estimate*---i.e., the difference between the average of `vote_prefer` variable in the two conditions---bounces around due to simple randomness.  That is, we want to see how the the luck of which subjects happen to get put into the treatment group; i.e., noise.

**You don't need to be able to write the code below on your own, but I do want to you to understand what it does.**

In [ ]:
# Function to re-randomize, or "shuffle", the treatment variable
re.randomize <- function(input.data) {
    input.data$treat <- sample(input.data$treat)
    return(input.data)
}

set.seed(123456)
reshuffle1 <- head(re.randomize(data_dh))
head(reshuffle1) # print the top of the resulting dataset


In [ ]:
reshuffle2 <- head(re.randomize(data_dh))
head(reshuffle2) # print the top of the resulting dataset


In [ ]:
reshuffle3 <- head(re.randomize(data_dh))
head(reshuffle3) # print the top of the resulting dataset

We can also just focus on the treatment column in each of these "simulated" data sets. Notice they are different each time:

In [ ]:
reshuffle1$treat
reshuffle2$treat
reshuffle3$treat

In [ ]:
# Function to compute the effect of the coethnic cousin treatment, relative to the noncoethnic noncousin condition
compute.cousincoethnic.effect <- function(input.data) {
    estimate <- difference_in_means(vote_prefer ~ treat, input.data)$coefficients
    return(as.numeric(estimate))
}

# First let's use this function to print again the actual estimate in the Dunning and Harrison (2010) data
actual.estimate <- compute.cousincoethnic.effect(data_dh)
actual.estimate

Now let's reshuffle the treatment labels at random, as we did above, and calculate the treatment effect:

In [ ]:
set.seed(54321)
compute.cousincoethnic.effect(re.randomize(data_dh))

And do that a few more times...

In [ ]:
compute.cousincoethnic.effect(re.randomize(data_dh))
compute.cousincoethnic.effect(re.randomize(data_dh))
compute.cousincoethnic.effect(re.randomize(data_dh))

The answer bounces around every time we run this function. This shows you why we need to worry about noise. If we just took this data, randomly shuffled it into groups, and then looked at the outcomes, having done nothing, we would still see some positive estimates even though putting subjections in the "treatment" group obviously did nothing in these simulations.

Is this what happened in Dunning and Harrison's study?? Does the co-ethnic cousin treatment have no effect on the outcomes---but we just happened to randomize subjects likely to want to vote for the candidate anyway into the treatment group??

Let's simulate 10,000 example experiments and see how big these estimates might get...

In [ ]:
# It will take a few seconds for this code to finish running
simulations <- replicate(10000, compute.cousincoethnic.effect(re.randomize(data_dh)))
head(simulations)

In [ ]:
# Let's plot a histogram of all the estimates using the hist() function you learned
# We'll also plot a red line at the actual value of the estimated effect in the real data, that is, 1.09

hist(as.numeric(simulations), breaks=15, main="Histogram of simulated estimates, and actual estimate", xlab="Estimates", xlim=c(-1.5, 1.5)) 
abline(v=actual.estimate, col = "red") 


The spread or width of this distribution shows you how large estimates are that we would see by chance.

We measure the spread of this distribution using the standard error:

In [ ]:
sd(simulations)

Later, we'll discuss how to assess formally how likely it is that an estimate of 1.09 would occur by chance if there were no true effect.  But the histogram here suggests it is very unlikely---the actual estimate is far from the distribution of values that can occur by chance when the treatment effect is zero.

## Good News: `difference_in_means()` tells you the standard error

You don't need to do this! The `difference_in_means()` function tells us the standard error. Let's run that code again to see:

In [ ]:
difference_in_means(vote_prefer ~ treat, data_dh)

The standard error is printed under the `Std. Error` heading. In this case, it's 0.1979227.

Note that 0.1979 $\approx$ 0.20 is slightly different than the standard error of 0.21 that we calculated using `sd(simulations)` above.  That's mainly because there's a little random error in our simulations... Don't worry much about that small difference now.

In [ ]:
# How big is the standard error relative to the estimte?
1.090944 / 0.1979227 # divide estimate / standard.error

This is called a _t_-statistic. It's the ratio of the estimate and the standard error:

$t = \frac{\text{Estimate}}{\text{Standard Error}}$

And `difference_in_means()` tells us this, too! (See 5.511972 above under `t value`.)

**The _t_-statistic is a way to measure how likely it is that an estimate of the size we saw would arise by chance even if the treatment had no effect.**

By convention, **we call an estimate _statistically significant_ if the _t_-statistic is larger than 1.96 (either lower than -1.96 or larger than 1.96).** Smaller than that (closer to zero), and we generally conclude that our estimate could have arisen by chance. In the Dunning and Harrison (2010) study, the _t_-statistic of 5.5 is far above 1.96 so the estimate is statistically significant.

We'll say more about how to use _t_-statistic and what statistical significance means in a future lecture.

## Formula for the Standard Error

You won't need to memorize this, but it turns out there is a formula that tells us what a standard error will be in an experiment.

Suppose we are comparing groups 1 and 2. Let:

- $s_1$ be the standard deviation of group 1
- $n_1$ be the sample size of group 1
- $s_2$ be the standard deviation of group 2
- $n_2$ be the sample size of group 2

Then this is the formula for the standard error in an experiment:

$ \text{Standard Error} = \sqrt{\frac{s^2_1}{n_1} + \frac{s^2_2}{n_2}} $

When the standard deviations are the same (so $s = s_1 = s_2$), this simplifes to:

$ \text{Standard Error} = s \sqrt{(\frac{1}{n_1} + \frac{1}{n_2})} $

And, when the two groups are *also* the same size (let $N = n_1 + n_2$ if $n_1 = n_2$), this simplifies further to:

$ \text{Standard Error} = 2 s \sqrt{\frac{1}{N}} $

What does this mean? For example:

- If standard deviation of the outcome variable doubles, the standard error doubles.
- If the sample size of a study is cut in half, the standard error increases by a factor of $\sqrt{2} \approx 1.41$. (E.g., if the standard error were $X$ before the sample size were cut in half, it would be $X \sqrt{2}$ after the sample size were cut in half.)
- If the sample size of the study doubles, the standard error decreases by a factor of $\sqrt{\frac{1}{2}} \approx .707$. (E.g., if the standard error were $X$ before the sample size were doubled, it would be $\frac{X}{\sqrt{2}}$ after the sample size were doubled.)
- In order to make the standard error half the size, the sample size must increase by a factor of 4 (i.e., quadruple).

**What I want you to remember**:

- The standard error goes up with the standard deviation or spread of the outcome variable.
- The standard error goes down as the sample size goes up.
- Although an experiment is more precise (i.e., the standard error is smaller) when the sample size is larger, it follows a square root, so we have to quadruple the sample size to cut the standard error in half (since $\sqrt{\frac{1}{4}} = \frac{1}{2}$). Likewise, in order to make the standard error of the treatment effect $\frac{1}{x}$ the size, the study needs to be $x^2$ times larger.
- In an experiment with a fixed *overall* sample size, the more similar in size that two groups are, the more precise the experiment will be. The most precise experiment at a fixed sample size will assign the treatment and control groups with equal probabilities (i.e., 50\% each).

## Reviewing New Terms/Concepts

- **True Average Treatment Effect**: If we could see all the potential outcomes, the actual truth about what the causal effect of a treatment is.
- **Estimate**: From a particular study run a particular time, our best guess of what the true average treatment effect is.
- **Bias**: When a study's estimates are systematically wrong in a particular direction; e.g., because of omitted variable bias. Experiments have no bias. If a study design is biased, it would be wrong even if its sample size were infinitely large.
- **Noise**: Because of random chance, a study's estimate differs from the truth, even though it is on average correct. If a study had an infinitely large sample size, it would have no noise.
- **Standard Error**: A way of measuring *how much* a study's estimate will differ from the truth (and between different runs of the same experiment) because of random chance. I.e., a measure of how much noise there is in an experiment.
- **t-statistic**: Defined as the estimate divided by the standard error. Gives an indication of how likely a study's result is to have arisen by chance. (More soon on how to use this.)